# *DELETE COLLECTION*

In [14]:
import chromadb
import os
from dotenv import load_dotenv

# 1. Load config environment
load_dotenv()

# SESUAIKAN PATH INI DENGAN CONFIG ANDA
# Default biasanya "./chroma_db" atau sesuai setting di .env Anda
CHROMA_PATH = os.getenv("CHROMA_DB_DIR", "./chroma_db") 
TARGET_COLLECTION = os.getenv("CHROMA_COLLECTION_NAME", "classicmodels") # Ganti nama collection jika perlu

def manage_chroma():
    # Cek apakah folder database ada
    if not os.path.exists(CHROMA_PATH):
        print(f"⚠️  Folder Chroma tidak ditemukan di: {os.path.abspath(CHROMA_PATH)}")
        return

    print(f"🔌 Connecting to ChromaDB at: {os.path.abspath(CHROMA_PATH)}")
    
    try:
        client = chromadb.PersistentClient(path=CHROMA_PATH)
    except Exception as e:
        print(f"❌ Gagal connect ke Chroma: {e}")
        return

    # ==========================================
    # 1. CARA CEK COLLECTION (LISTING)
    # ==========================================
    collections = client.list_collections()
    
    print("\n📋 EXISTING COLLECTIONS:")
    if not collections:
        print("   (Database kosong, tidak ada collection)")
    else:
        existing_names = [c.name for c in collections]
        for name in existing_names:
            print(f"   - {name}")

    # ==========================================
    # 2. CARA HAPUS COLLECTION
    # ==========================================
    print(f"\n🗑️  Mencoba menghapus collection target: '{TARGET_COLLECTION}'...")
    
    # Cek apakah target ada di list
    if any(c.name == TARGET_COLLECTION for c in collections):
        try:
            client.delete_collection(name=TARGET_COLLECTION)
            print(f"✅ SUKSES! Collection '{TARGET_COLLECTION}' berhasil dihapus.")
            print("   Sekarang Anda bisa menjalankan ingestion ulang dengan bersih.")
        except Exception as e:
            print(f"❌ Error saat menghapus: {e}")
    else:
        print(f"⚠️  Collection '{TARGET_COLLECTION}' tidak ditemukan. Tidak ada yang perlu dihapus.")

if __name__ == "__main__":
    manage_chroma()

🔌 Connecting to ChromaDB at: /app/chroma_db

📋 EXISTING COLLECTIONS:
   (Database kosong, tidak ada collection)

🗑️  Mencoba menghapus collection target: 'classicmodels'...
⚠️  Collection 'classicmodels' tidak ditemukan. Tidak ada yang perlu dihapus.


# *CHECK COLLECTION TIMESTAMP*

In [10]:
from pathlib import Path
import json
import os
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv, find_dotenv

UTC_PLUS_7 = timezone(timedelta(hours=7))

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=False)

env_dir = Path(env_path).resolve().parent if env_path else Path.cwd()

raw_chroma_dir = os.getenv("CHROMA_DIR", "./chroma_db").strip()
collection_name = os.getenv("COLLECTION_NAME", "classicmodels").strip()

chroma_dir = Path(raw_chroma_dir)
if not chroma_dir.is_absolute():
    chroma_dir = (env_dir / chroma_dir).resolve()

stamp_path = chroma_dir / f".{collection_name}.dense_source_stamp.json"
db_path = chroma_dir / "chroma.sqlite3"

print("ENV_PATH         =", env_path)
print("ENV_DIR          =", env_dir)
print("CHROMA_DIR       =", chroma_dir)
print("COLLECTION_NAME  =", collection_name)
print("STAMP_PATH       =", stamp_path)
print("DB_PATH          =", db_path)
print()

if stamp_path.exists():
    payload = json.loads(stamp_path.read_text(encoding="utf-8"))
    stamp_mtime = datetime.fromtimestamp(stamp_path.stat().st_mtime, tz=UTC_PLUS_7)
    print("Dense source stamp exists")
    print("Stamp modified UTC+7 :", stamp_mtime.strftime("%Y-%m-%d %H:%M:%S"))
    print("Signature            :", payload.get("signature"))
    print("Paths                :", payload.get("paths"))
else:
    print("Dense source stamp does not exist")

print()

if db_path.exists():
    db_mtime = datetime.fromtimestamp(db_path.stat().st_mtime, tz=UTC_PLUS_7)
    print("Chroma sqlite exists")
    print("DB modified UTC+7    :", db_mtime.strftime("%Y-%m-%d %H:%M:%S"))
else:
    print("chroma.sqlite3 does not exist")

ENV_PATH         = /app/.env
ENV_DIR          = /app
CHROMA_DIR       = /app/chroma_db
COLLECTION_NAME  = classicmodels
STAMP_PATH       = /app/chroma_db/.classicmodels.dense_source_stamp.json
DB_PATH          = /app/chroma_db/chroma.sqlite3

Dense source stamp exists
Stamp modified UTC+7 : 2026-05-19 07:17:57
Signature            : /app/metadata_jsonl/classicmodels_dense.jsonl:869a3663beb427f50cb1cce74590f997fee8920dbb38158169ff2367aec2866f
Paths                : ['/app/metadata_jsonl/classicmodels_dense.jsonl']

Chroma sqlite exists
DB modified UTC+7    : 2026-05-19 07:20:48
